# Time series models (non-DL baselines)

In [27]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import time

np.random.seed(42)

In [2]:
DATA_DIR = "../data/"
HORIZON = 28
SEASON = 7

In [ ]:
sales = pd.read_csv(DATA_DIR + "sales_train_evaluation.csv")
day_cols = [c for c in sales.columns if c.startswith("d_")]

In [26]:
# build 1 time series model per item for the top store CA_3
sales_ca3 = sales[sales["store_id"] == "CA_3"].reset_index(drop=True)
series_matrix = sales_ca3[day_cols].values.astype(np.float32)
ids = sales_ca3["id"].values

In [11]:
series_matrix
# 3049 items, 1941 days 

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 2., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 5., 4., 0.]], shape=(3049, 1941), dtype=float32)

In [12]:
# keep the last 28 days as test data
y_train = series_matrix[:, :-HORIZON]
y_test = series_matrix[:, -HORIZON:]

# Seasonal Naive

In [ ]:
# Model 1: seasonal naive
last_week = y_train[:, -SEASON:]
# just repeat the last 7 days 4 times  
y_pred = np.tile(last_week, (HORIZON // SEASON + 1))[:, :HORIZON]

In [18]:
forecast_cols = [f"F{i}" for i in range(1, HORIZON + 1)]
results_naive = pd.DataFrame(y_pred, columns=forecast_cols)
# append item ID
results_naive.insert(0, "id", ids) 

In [19]:
# evaluate model 1
rmse_naive = np.sqrt(((y_pred - y_test) ** 2).mean())
print(f"Seasonal Naive RMSE: {rmse_naive:.2f}")
print(results_naive.head())
# results.to_csv("seasonal_naive_forecasts.csv", index=False)

Seasonal Naive RMSE: 3.32
                              id    F1   F2   F3   F4   F5    F6   F7    F8  \
0  HOBBIES_1_001_CA_3_evaluation   0.0  1.0  1.0  1.0  0.0   3.0  3.0   0.0   
1  HOBBIES_1_002_CA_3_evaluation   2.0  0.0  0.0  0.0  0.0   1.0  0.0   2.0   
2  HOBBIES_1_003_CA_3_evaluation   0.0  0.0  0.0  0.0  0.0   1.0  1.0   0.0   
3  HOBBIES_1_004_CA_3_evaluation  15.0  3.0  0.0  3.0  4.0  12.0  3.0  15.0   
4  HOBBIES_1_005_CA_3_evaluation   1.0  2.0  0.0  0.0  0.0   5.0  1.0   1.0   

    F9  ...  F19   F20  F21   F22  F23  F24  F25  F26   F27  F28  
0  1.0  ...  0.0   3.0  3.0   0.0  1.0  1.0  1.0  0.0   3.0  3.0  
1  0.0  ...  0.0   1.0  0.0   2.0  0.0  0.0  0.0  0.0   1.0  0.0  
2  0.0  ...  0.0   1.0  1.0   0.0  0.0  0.0  0.0  0.0   1.0  1.0  
3  3.0  ...  4.0  12.0  3.0  15.0  3.0  0.0  3.0  4.0  12.0  3.0  
4  2.0  ...  0.0   5.0  1.0   1.0  2.0  0.0  0.0  0.0   5.0  1.0  

[5 rows x 29 columns]


# SARIMA

In [31]:
# Model 2: SARIMA
sales_small = sales_ca3.sample(30)
results = []
rmses = []
fallback_count = 0

t0 = time.time()

# 1 SARIMA model per item 

for i, row in sales_small.iterrows():
    train = row[day_cols].values[:-HORIZON].astype(float)
    actual = row[day_cols].values[-HORIZON:].astype(float)
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("error")
            model = SARIMAX(train, order=(0,1,0), seasonal_order=(0,1,0,7),
                            enforce_stationarity=False, enforce_invertibility=False)
            pred = model.fit(disp=False, maxiter=200).forecast(steps=HORIZON)
            pred = np.clip(pred, 0, None)
    except Exception:
        pred = np.tile(train[-7:], HORIZON // 7 + 1)[:HORIZON]
        fallback_count += 1

    rmses.append(np.sqrt(((pred - actual) ** 2).mean()))
    results.append([row["id"]] + list(pred))
    print(f"{i}/{len(sales)} | RMSE: {np.mean(rmses):.2f} | Fallbacks: {fallback_count}")

df = pd.DataFrame(results, columns=["id"] + forecast_cols)
df.to_csv("sarima_forecasts_CA3.csv", index=False)
print(f"\nFinal RMSE: {np.mean(rmses):.2f} | Fallbacks: {fallback_count}/{len(sales_small)}")

t1 = time.time()
print(f"\nSARIMA modeling completed in {(t1 - t0) / 60:.2f} minutes.")

1435/30490 | RMSE: 0.33 | Fallbacks: 1
357/30490 | RMSE: 0.50 | Fallbacks: 2
2664/30490 | RMSE: 1.29 | Fallbacks: 3
1037/30490 | RMSE: 1.51 | Fallbacks: 4
82/30490 | RMSE: 1.53 | Fallbacks: 5
905/30490 | RMSE: 1.91 | Fallbacks: 6
2373/30490 | RMSE: 1.91 | Fallbacks: 7
2666/30490 | RMSE: 1.76 | Fallbacks: 8
2316/30490 | RMSE: 1.79 | Fallbacks: 9
1328/30490 | RMSE: 1.66 | Fallbacks: 10
2968/30490 | RMSE: 2.17 | Fallbacks: 11
311/30490 | RMSE: 2.15 | Fallbacks: 12
2248/30490 | RMSE: 2.13 | Fallbacks: 13
2455/30490 | RMSE: 2.29 | Fallbacks: 14
1073/30490 | RMSE: 2.25 | Fallbacks: 15
2284/30490 | RMSE: 2.23 | Fallbacks: 16
351/30490 | RMSE: 2.11 | Fallbacks: 17
1140/30490 | RMSE: 2.22 | Fallbacks: 18
1915/30490 | RMSE: 2.17 | Fallbacks: 19
1218/30490 | RMSE: 2.12 | Fallbacks: 20
2213/30490 | RMSE: 2.13 | Fallbacks: 21
2144/30490 | RMSE: 2.20 | Fallbacks: 22
2890/30490 | RMSE: 2.21 | Fallbacks: 23
200/30490 | RMSE: 2.16 | Fallbacks: 24
1066/30490 | RMSE: 2.13 | Fallbacks: 25
218/30490 | RMSE

In [ ]:
df_sarima = pd.DataFrame(results_sarima, columns=["id"] + forecast_cols)
# df.to_csv("sarima_forecasts_CA3.csv", index=False)
print(f"\nFinal Mean RMSE (CA_3): {np.mean(rmses_sarima):.2f}")

Issues:
- Many sparse series so SARIMA failed to converge. SARIMA is unsuitable for this data

# End